In [3]:
from google.cloud import bigquery
from datetime import datetime, timedelta
import random

In [4]:
project_id = 'pcloud2026'
region = 'europe-west8'
db_id = 'ais'
table_id = 'ais2023_1'
client = bigquery.Client.from_service_account_json('../secret.json')

In [5]:
# query bigqeury to group by MMSI and compute the maximum delta of LAT in each group
#order from the highest to the lowest
#limit 100
querys = f"""
    SELECT MMSI, MAX(LAT) - MIN(LAT) AS delta_lat
    FROM `{project_id}.{db_id}.{table_id}`
    GROUP BY MMSI
    ORDER BY delta_lat DESC
    LIMIT 100
    """ 
df = client.query(querys).to_dataframe()
df

,MMSI,delta_lat
0,367354090,62.37766
1,338209606,60.89126
2,366933870,57.00005
3,338424122,55.92408
4,368281980,42.66238
...,...,...
95,566167000,23.91041
96,636092932,23.86376
97,229395000,23.74333
98,374077000,23.64844


In [6]:
querys = f"""
    SELECT *
    FROM `{project_id}.{db_id}.{table_id}`
    WHERE MMSI = 636014430
    """

df = client.query(querys).to_dataframe()
df

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading
0,636014430,1673748958000000000,39.53098,-125.07049,2.6,269.4,287.0
1,636014430,1673997032000000000,48.38179,-124.40448,3.8,115.6,121.0
2,636014430,1673768251000000000,39.77589,-125.70092,6.9,299.9,294.0
3,636014430,1674369750000000000,48.39563,-123.26002,7.9,238.4,240.0
4,636014430,1674012483000000000,48.22544,-123.69780,8.0,83.8,87.0
...,...,...,...,...,...,...,...
18259,636014430,1673976183000000000,48.11656,-125.21846,11.1,18.0,21.0
18260,636014430,1673557104000000000,33.94970,-119.00188,11.1,299.7,302.0
18261,636014430,1674911422000000000,27.54667,-115.77680,11.1,140.7,138.0
18262,636014430,1673961703000000000,47.36735,-125.33306,11.1,1.2,6.0


In [7]:
#sample 100 rows from df and sort by BaseDateTime
df_sample = df.sample(n=100, random_state=42).sort_values(by='BaseDateTime')
df_sample.head()

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading
4104,636014430,1672587040000000000,33.76494,-118.26002,0.0,100.9,177.0
7650,636014430,1672643143000000000,33.73613,-118.22414,0.1,103.2,332.0
9092,636014430,1672645841000000000,33.73664,-118.22363,0.3,18.7,298.0
8062,636014430,1672673924000000000,33.73618,-118.22526,0.1,221.3,28.0
5857,636014430,1672723793000000000,33.73662,-118.22582,0.1,191.6,56.0


In [9]:
# create a plot with folium with the lat and lon of the dataframe
import folium
m = folium.Map(location=[df_sample['LAT'].mean(), df_sample['LON'].mean()], zoom_start=10)
for index, row in df_sample.iterrows():
    folium.Marker(location=[row['LAT'], row['LON']], popup=row['MMSI']).add_to(m)       
m